In [0]:
%pip install langgraph langchain langchain-community langsmith langchain-google-genai pygame


Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 135.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.9/611.9 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-d2d8554d-0afe-475b-bc5a-c11489ef92e2
    Can't uninstall 'typing_extensions'. No files

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import uuid
import operator
import re
import subprocess
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage,ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.types import interrupt
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

In [0]:
os.environ["GOOGLE_API_KEY"] = dbutils.secrets.get(scope="dino-runner-secrets", key="google-api-key")
os.environ["LANGCHAIN_API_KEY"] = dbutils.secrets.get(scope="dino-runner-secrets", key="langsmith-api-key")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "dino-runner-multiagent"

### Handling Provider-Specific Message Formats

Different LLM providers structure their response content differently -
OpenAI returns plain strings, while Gemini returns a list of content
blocks. A shared get_text() helper normalizes this across the pipeline,
so every node can reliably extract plain text regardless of which
provider generated it.

In [0]:
def get_text(message) -> str:
    """
    Extracts plain text from a message's .content, handling both
    OpenAI-style (content is a string) and Gemini-style (content is a
    list of dicts like [{'type': 'text', 'text': '...'}]) formats.
    """
    content = message.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        # Join all text-type parts together, ignoring non-text parts
        return "".join(
            part.get("text", "") for part in content if isinstance(part, dict)
        )
    return str(content)

## System Memory — Defining GameState

Before any agent can act, the pipeline needs a shared memory structure that
persists across every node in the graph. This is `GameState` — a single
object passed from agent to agent, where each one reads what previous
agents wrote and adds its own contribution.

Fields that accumulate history (`director_messages`, `architect_messages`,
`engineer_code`, `qa_feedback`, `iteration_score`) use **reducers** so that
new values are appended rather than overwriting what came before. This
preserves a full record of every iteration — every version of the code,
every QA report, every score — instead of only keeping the latest one.

Fields that only ever need their current value (`current_actor`,
`iteration`, `file_saved`) use plain types with default overwrite behavior,
since there's no need to track their history.

In [0]:
class GameState(TypedDict):
    director_messages: Annotated[list, add_messages]
    architect_messages: Annotated[list, add_messages]
    engineer_code: Annotated[list, add_messages]
    qa_feedback: Annotated[list, add_messages]
    current_actor: str
    iteration: int
    iteration_score: Annotated[list[int], operator.add]
    file_saved: bool

## Defining the Agent Nodes

With shared memory in place, the next step is to build the actual agents
that will read from and write to it. Each agent is implemented as a single
Python function — a **node** — that takes in the current `GameState`,
performs its specific job, and returns the fields it wants to update.

Six nodes make up the pipeline: Director, Architect, Engineer, Execution
Manager, QA Engineer, and Scorer. Some of these call an LLM to reason or
generate content (Architect, Engineer, QA, Scorer); others are plain Python
logic with no model call involved (Director, Execution Manager). Together,
they form a loop where code gets written, tested, evaluated, and — if
needed — sent back for revision.

### Director Node

The Director represents the starting point of the pipeline — the initial
objective for what should be built. It requires no LLM call; it simply
packages the request as a message and hands control to the Architect.

In [0]:
def director_node(state: GameState):
    # In a real run, this could come from user input;
    # for now we hardcode the initial objective. The exact required
    # features are spelled out here (not left implicit) so every
    # downstream agent - Architect, Engineer, QA - is grounded in the
    # same concrete checklist instead of a vague "build a dino game".
    prompt = (
        "Build a Dino Runner game (Chrome Dino style) in Python using pygame. "
        "It must include ALL of the following, exactly:\n"
        "1. Flying obstacles (e.g. Pterodactyls) at variable heights.\n"
        "2. Ground obstacles (e.g. Cacti).\n"
        "3. Accurate jumping and falling physics (gravity, velocity, a landing check).\n"
        "4. Jumping mapped to one key and ducking mapped to a different key, "
        "with a visibly smaller hitbox/sprite while ducking.\n"
        "5. A functional high-score tracking system (persists the best run's "
        "score within the session and displays both current score and high score)."
    )
    return {
        "director_messages": [HumanMessage(content=prompt)],
        "current_actor": "architect"
    }


### Architect Node

The Architect reads the Director's request and produces a high-level
system design — a breakdown of the components the game will need (e.g.
Player class, obstacle spawner, collision detection, scoring). This design
becomes the blueprint the Engineer will translate into actual code.


In [0]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.2)

def architect_node(state: GameState):
    director_request = state["director_messages"][-1].content
    
    system_prompt = (
        "You are a software architect. Break down the following game request "
        "into a clear list of system components and features needed "
        "(e.g. Player class, obstacle spawner, collision detection, scoring system)."
    )
    
    response = llm.invoke([
        ("system", system_prompt),
        ("human", director_request)
    ])
    
    return {
        "architect_messages": [AIMessage(content=response.content)],
        "current_actor": "engineer"
    }

### Engineer Node

The Engineer turns a design into working Python code. On the first pass,
it builds the game from the Architect's design. On every pass after that,
it instead reads the previous code together with the QA Engineer's
feedback, and returns a corrected full version of the code — this is the
node that actually improves with each loop of the pipeline.

In [0]:
def engineer_node(state: GameState):
    design = get_text(state["architect_messages"][-1])
    
    if state["qa_feedback"]:
        latest_feedback = get_text(state["qa_feedback"][-1])
        instruction = (
            f"Here is the current game code and QA feedback on it. "
            f"Make ONLY the minimal changes needed to fix the specific issues "
            f"listed. Do NOT rewrite unrelated parts of the code, change the "
            f"art style, rename classes, or restructure the file. Preserve "
            f"everything that isn't broken. Return the FULL file with your "
            f"targeted fixes applied.\n\n"
            f"Current code:\n{get_text(state['engineer_code'][-1])}\n\n"
            f"QA Feedback:\n{latest_feedback}"
        )
    else:
        instruction = f"Write a complete Python Dino Runner game based on this design:\n{design}"
    
    system_prompt = (
        "You are a Python game developer using pygame. "
        "Always return ONLY the complete, runnable Python code — "
        "no explanations, no markdown code fences, just raw code."
    )
    
    response = llm.invoke([
        ("system", system_prompt),
        ("human", instruction)
    ])
    
    return {
        "engineer_code": [AIMessage(content=get_text(response))],
        "current_actor": "execution_manager",
        "iteration": state["iteration"] + 1
    }

### File I/O and Execution Nodes

Once the Engineer produces code, it needs to be written to disk and
actually run to see if it works. This is split into two responsibilities:
saving the code to a `.py` file, and executing that file as a separate
process to capture its output or errors.

Running arbitrary AI-generated code automatically carries risk, so a
human-in-the-loop checkpoint is placed before execution — the pipeline
pauses using LangGraph's interrupt() mechanism and waits for explicit
approval before the code is actually run.

In [0]:
def save_code_node(state: GameState):
    """
    Writes the Engineer's latest generated code to a local .py file.
    Plain file-write, no LLM involved, so no message object is needed
    for this node's own action.
    """
    # Handle case where content might be a list or string
    latest_code = get_text(state["engineer_code"][-1])

    with open("dino_runner.py", "w") as f:
        f.write(latest_code)

    return {
        "file_saved": True,
        "current_actor": "execution_manager"
    }


def execution_node(state: GameState):
    """
    Pauses for human approval using LangGraph's interrupt(), then runs
    the saved code as a subprocess for a short window. Since a real
    game loop never exits on its own in a headless environment, a
    timeout here is treated as a SUCCESS signal (code launched and
    ran without crashing), not a failure - only an actual exception/
    traceback counts as something QA needs to flag.

    Note: notebook/CI environments (e.g. Databricks) have no display
    server. Without SDL_VIDEODRIVER=dummy, pygame.display.set_mode()
    would raise immediately - not time out - which would incorrectly
    look like a real bug to QA. We run the subprocess with a "dummy"
    video driver so the game can actually initialize headlessly; the
    real, visual demo (Task 5) should still be run locally with a
    normal display, outside this harness.
    """
    approval = interrupt(
        {"question": "Approve running the generated code?(y/n)", "code_preview": get_text(state["engineer_code"][-1])[:300]}
    )

    if approval != "y":
        return {
            "qa_feedback": [ToolMessage(content="Execution was rejected by human reviewer.", tool_call_id="execution_manager")],
            "current_actor": "qa"
        }

    headless_env = dict(os.environ)
    headless_env["SDL_VIDEODRIVER"] = "dummy"
    headless_env["SDL_AUDIODRIVER"] = "dummy"

    try:
        result = subprocess.run(
            ["python", "dino_runner.py"],
            capture_output=True,
            text=True,
            timeout=8,  # short window - we only need to catch import/syntax errors
            env=headless_env
        )
        execution_log = f"STDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}"

    except subprocess.TimeoutExpired as e:
        # The process was still running after 8 seconds with no crash -
        # this means the game loop started successfully and is just
        # waiting for a window/input it will never get in this headless
        # environment. Treat this as evidence of a healthy launch.
        partial_output = (e.stdout or b"").decode() if e.stdout else ""
        partial_errors = (e.stderr or b"").decode() if e.stderr else ""
        execution_log = (
            f"Process ran for 8+ seconds without crashing (timed out as expected "
            f"for a headless game loop with no window to close).\n\n"
            f"Partial STDOUT:\n{partial_output}\n\nPartial STDERR:\n{partial_errors}"
        )

    return {
        "qa_feedback": [ToolMessage(content=execution_log, tool_call_id="execution_manager")],
        "current_actor": "qa"
    }


### QA and Scorer Nodes

With the code executed and its raw output captured, two more agents
interpret what happened. The QA Engineer reads the code, the design
requirements, and the execution output together, and writes an analysis
identifying bugs or missing features. The Scorer then reads that analysis
and distills it into a single quantitative score (1–10), giving the
human-in-the-loop a quick signal for whether another iteration is needed.

In [0]:
def qa_node(state: GameState):
    """
    Analyzes the latest code, design requirements, and execution output.
    Now explicitly calibrated to avoid manufacturing issues when the code
    is genuinely close to correct, and to prioritize crash/timeout signals
    over cosmetic nitpicks. Also grounded directly in the Director's
    original requirements (not just the Architect's paraphrase of them),
    so QA checks against the exact 5 required features from Task 5
    rather than whatever subset the Architect happened to restate.
    """
    code = get_text(state["engineer_code"][-1])
    original_requirements = get_text(state["director_messages"][-1])
    design = get_text(state["architect_messages"][-1])
    execution_log = get_text(state["qa_feedback"][-1])

    system_prompt = (
        "You are a QA engineer reviewing an AI-generated Python game. "
        "Compare the code and execution output against BOTH the original "
        "requirements and the architect's design. List concrete bugs, "
        "missing features, or crashes, and describe the fix needed in "
        "plain English for each one.\n\n"
        "Explicitly check for these 5 required features and call out by "
        "name any that are missing or broken: flying obstacles, ground "
        "obstacles, jump/fall physics, a distinct duck mechanic, and a "
        "working high-score tracker.\n\n"
        "Do NOT write or include any corrected code, patches, or full "
        "implementations — your job is diagnosis only.\n\n"
        "Calibration: if the code is functionally correct, runs without "
        "crashing, and meets the design requirements, say so explicitly and "
        "report FEWER than 2 issues — do not manufacture problems just to "
        "fill out the report. Only list genuine, verifiable issues. A "
        "'process timed out without crashing' execution result is a PASS, "
        "not a bug — headless environments have no window to close, so this "
        "is expected and should not be flagged as an issue."
    )

    instruction = (
        f"Original requirements:\n{original_requirements}\n\n"
        f"Design requirements:\n{design}\n\n"
        f"Code:\n{code}\n\n"
        f"Execution output:\n{execution_log}"
    )

    response = llm.invoke([
        ("system", system_prompt),
        ("human", instruction)
    ])

    return {
        "qa_feedback": [AIMessage(content=get_text(response))],
        "current_actor": "scorer"
    }

def scorer_node(state: GameState):
    """
    Converts QA's report into a 1-10 score, with explicit bands so the
    LLM isn't guessing blind. Score is judged on severity and coverage of
    remaining issues, not just "how many bullet points did QA write."

    Parsing is now defensive: even though the prompt asks for ONLY a
    number, LLMs occasionally add a stray word or period. A regex pulls
    the first integer out of the response instead of a bare int(), so a
    minor formatting slip can't crash the whole graph run.
    """
    qa_report = get_text(state["qa_feedback"][-1])

    system_prompt = (
        "You are a scoring engine. Read the QA report and output ONLY "
        "a single integer from 1 to 10 representing overall code health "
        "and feature completeness. No words, no explanation, just the number.\n\n"
        "Scoring bands:\n"
        "9-10: No meaningful bugs remain, all design requirements are met, "
        "QA explicitly says the code is functionally correct.\n"
        "7-8: Minor cosmetic or edge-case issues only, nothing that affects "
        "core gameplay or crashes the game.\n"
        "4-6: At least one functional bug affecting gameplay, but the game "
        "is playable and doesn't crash.\n"
        "1-3: The game crashes, fails to run, or is missing a core required "
        "feature (jumping, ducking, obstacles, or scoring).\n\n"
        "Do not default to the middle of the range — use the full scale "
        "based on actual severity."
    )

    response = llm.invoke([
        ("system", system_prompt),
        ("human", qa_report)
    ])

    raw = get_text(response).strip()
    match = re.search(r"\d+", raw)
    score = int(match.group()) if match else 5  # neutral fallback, never crash the graph
    score = max(1, min(10, score))  # clamp to the valid 1-10 range

    return {
        "iteration_score": [score],
        "current_actor": "human_review"
    }


## Building the Graph Structure

All six agents now exist as individual functions, but nothing connects
them yet. This step wires them together into an actual LangGraph
StateGraph — defining the order agents run in, and the one place where
the flow can branch: after scoring, based on human approval, either
looping back to the Engineer for another round of fixes, or finishing.

In [0]:
# Create the graph, telling it what shape the shared state is
# (the GameState we defined back in Task 1)
graph = StateGraph(GameState)

# Register every function as a "node" - just giving each one
# a name (a string) so we can refer to it when connecting things.
graph.add_node("director", director_node)
graph.add_node("architect", architect_node)
graph.add_node("engineer", engineer_node)
graph.add_node("save_code", save_code_node)
graph.add_node("execution", execution_node)
graph.add_node("qa", qa_node)
graph.add_node("scorer", scorer_node)

#  Tell the graph where to START - the very first node to run
graph.set_entry_point("director")

# Connect the simple, no-decision steps in a straight line.
# add_edge(A, B) just means "after A finishes, always go to B next."
graph.add_edge("director", "architect")
graph.add_edge("architect", "engineer")
graph.add_edge("engineer", "save_code")
graph.add_edge("save_code", "execution")
graph.add_edge("execution", "qa")
graph.add_edge("qa", "scorer")

# This is the ONE branching point - after scorer, we need to
# ask a human whether to loop back or stop, instead of always going
# to the same next node.

def route_after_scoring(state: GameState):
    """
    Decides what happens after scoring: ask the human, then return the
    NAME of the next node to go to (as a plain string). LangGraph reads
    this returned string and routes the graph there.
    """
    latest_score = state["iteration_score"][-1]

    # Pause here too, showing the human the score before deciding
    decision = interrupt(
        {"question": f"Current score: {latest_score}/10. Loop again for fixes, or finish?",
         "options": ["loop", "finish"]}
    )

    if decision == "finish":
        return "end"       # a plain label, mapped below to the real END
    else:
        return "engineer"  # send it back to the Engineer to try again

# add_conditional_edges needs: (1) which node this decision happens after,
# (2) the function that decides, (3) a mapping from that function's
# possible return values to actual node names / END.
graph.add_conditional_edges(
    "scorer",
    route_after_scoring,
    {
        "engineer": "engineer",  # if function returned "engineer", go there
        "end": END                # if function returned "end", finish the graph
    }
)

# Step 6: Compile the graph into a runnable object.
# checkpointer=MemorySaver() gives it memory ACROSS pauses - this is
# required for interrupt() to work, since the graph needs somewhere to
# save "where it stopped" while it's waiting for your input.
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

## Running the Graph

With the graph compiled, this step actually invokes it — streaming
output live so each agent's action is visible as it happens, rather
than waiting silently for a single final result. Because two nodes
in the pipeline pause on interrupt() (execution approval, and the
loop-or-finish decision after scoring), running the graph also means
handling those pauses: reading the interrupt's payload, collecting the
human's decision, and resuming the graph with that decision so it can
continue from exactly where it stopped.

In [0]:
# A unique ID for this particular run of the graph - required so
# MemorySaver knows which saved checkpoint belongs to this run.
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

# GameState needs starting values for every field, even empty ones,
# since nodes will read from them (e.g. qa_feedback starts empty).
initial_state = {
    "director_messages": [],
    "architect_messages": [],
    "engineer_code": [],
    "qa_feedback": [],
    "current_actor": "director",
    "iteration": 0,
    "iteration_score": [],
    "file_saved": False
}


In [0]:
def run_graph_with_interrupts(app, initial_state, config):
    """
    Runs the graph, and every time it pauses on an interrupt(), prompts
    the human for input and resumes - repeating until the graph reaches
    END with no more pauses left.
    """
    stream_input = initial_state

    while True:
        interrupted = False

        for event in app.stream(stream_input, config=config):
            print(event)

            if "__interrupt__" in event:
                interrupted = True
                interrupt_data = event["__interrupt__"][0].value
                print("PAUSED:", interrupt_data)

                user_input = input("Your response: ")
                # Set up what to feed into the NEXT loop iteration
                stream_input = Command(resume=user_input)

        if not interrupted:
            # No interrupt was hit during this pass - graph finished naturally
            break

In [0]:
run_graph_with_interrupts(app, initial_state, config)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'director': {'director_messages': [HumanMessage(content="Build a Dino Runner game (Chrome Dino style) in Python using pygame. It must include ALL of the following, exactly:\n1. Flying obstacles (e.g. Pterodactyls) at variable heights.\n2. Ground obstacles (e.g. Cacti).\n3. Accurate jumping and falling physics (gravity, velocity, a landing check).\n4. Jumping mapped to one key and ducking mapped to a different key, with a visibly smaller hitbox/sprite while ducking.\n5. A functional high-score tracking system (persists the best run's score within the session and displays both current score and high score).", additional_kwargs={}, response_metadata={}, id='1f4d05dd-5f0a-407e-9de0-20187299931e')], 'current_actor': 'architect'}}
{'architect': {'architect_messages': [AIMessage(content=[{'type': 'text', 'text': 'As a software architect, I have broken down the Chrome Dino Runner game into a modular, object-oriented system architecture. This design ensures clean separation of concerns, making

Your response:  y

{'execution': {'qa_feedback': [ToolMessage(content='Process ran for 8+ seconds without crashing (timed out as expected for a headless game loop with no window to close).\n\nPartial STDOUT:\n\n\nPartial STDERR:\n', id='16fc0c18-b4f8-4ccc-8bce-cd9dc18b4685', tool_call_id='execution_manager')], 'current_actor': 'qa'}}
{'qa': {'qa_feedback': [AIMessage(content='### QA Review\n\nThe code runs without crashing, implements all core architectural components, and successfully meets the primary requirements. However, there is a significant gameplay physics/collision bug regarding the flying obstacles.\n\n---\n\n### **Required Features Status**\n\n1. **Flying Obstacles (Pterodactyls):** **Present.** Spawns at variable heights with a 2-frame flapping animation.\n2. **Ground Obstacles (Cacti):** **Present.** Spawns small/large cacti in single or grouped configurations.\n3. **Jump/Fall Physics:** **Present.** Implements gravity, velocity, fast-falling when pressing down, and a ground landing check.\

Your response:  loop

{'scorer': {'iteration_score': [5], 'current_actor': 'human_review'}}
{'engineer': {'engineer_code': [AIMessage(content='import pygame\nimport random\nimport sys\n\n# Initialize Pygame\npygame.init()\npygame.font.init()\n\n# Constants\nSCREEN_WIDTH = 800\nSCREEN_HEIGHT = 400\nFPS = 60\nGROUND_Y = 320\n\n# Colors\nCOLOR_BG = (247, 247, 247)\nCOLOR_PRIMARY = (83, 83, 83)\nCOLOR_LIGHT = (220, 220, 220)\n\n# Setup Display\nscreen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))\npygame.display.set_caption("Dino Runner")\nclock = pygame.time.Clock()\n\n# --- Procedural Sprite Generation ---\n\ndef create_dino_sprite(state, frame, color=COLOR_PRIMARY):\n    """Generates Dino sprites procedurally to avoid external asset dependencies."""\n    if "duck" in state:\n        surf = pygame.Surface((59, 30), pygame.SRCALPHA)\n        # Body\n        pygame.draw.rect(surf, color, (10, 10, 38, 14))\n        # Head\n        pygame.draw.rect(surf, color, (44, 6, 15, 10))\n        # Beak/Snout\n 

Your response:  y

{'execution': {'qa_feedback': [ToolMessage(content='Process ran for 8+ seconds without crashing (timed out as expected for a headless game loop with no window to close).\n\nPartial STDOUT:\n\n\nPartial STDERR:\n', id='910e3acd-f626-4422-b916-5857fa74bc81', tool_call_id='execution_manager')], 'current_actor': 'qa'}}
{'qa': {'qa_feedback': [AIMessage(content="The code is functionally correct, runs without crashing, and fully meets both the original requirements and the architect's design. \n\n### **Required Features Verification:**\n*   **Flying obstacles:** Fully implemented via the `Pterodactyl` class, which spawns at three distinct, well-calibrated heights (`low`, `mid`, and `high`) with a 2-frame wing-flapping animation.\n*   **Ground obstacles:** Fully implemented via the `Cactus` class, supporting randomized sizes (small/large) and grouping counts (1, 2, or 3).\n*   **Jump/fall physics:** Accurately implemented with gravity, initial jump velocity, fast-falling when pressing down in

Your response:  finish

{'scorer': {'iteration_score': [10], 'current_actor': 'human_review'}}


## What each node adds to state, precisely
- Director — writes director_messages (the initial prompt). No LLM.
- Architect — reads director_messages, calls the LLM, writes architect_messages.
- Engineer — reads architect_messages (first pass) or engineer_code + qa_feedback (later passes), calls the LLM, writes engineer_code, bumps iteration.
- Save code — reads engineer_code, writes the .py file to disk, sets file_saved. No LLM.
- Execution — pauses (interrupt) for your approval, runs the file as a subprocess, writes the raw output into qa_feedback as a placeholder. No LLM.
- QA — reads engineer_code + architect_messages + that raw execution output, calls the LLM, overwrites the placeholder by appending its actual analysis to qa_feedback.
- Scorer — reads the latest qa_feedback, calls the LLM, appends a number to iteration_score.
- Human review — the conditional edge: pauses again, asks you loop/finish, and either routes back to Engineer or to END.